# S&P 500 Stock Market Performance & Business Intelligence (2014–2017)
**Analyst:** Olabode Goodness Daniel (OGD Analytics) | **Track:** Data Analytics (Track A) | **Programme:** AnalystLab Africa Experience Lab

## 1. Business Problem & Objectives
Financial researchers and market analysts need structured, validated insights across 505 S&P 500 equities from 2014 to 2017 (497,472 daily trading records). Rather than relying on raw nominal share prices—which distort cross-company comparisons—this notebook inspects data quality, engineers per-symbol return and volatility metrics, evaluates market trends across 8 core business questions, and compares 6 representative focus stocks (`NVDA`, `AMZN`, `AAPL`, `BAC`, `JNJ`, `CHK`).

**Key Scope & Historical Limitations:**
- Dataset covers **January 2, 2014 to December 29, 2017** (1,007 trading days) and does not reflect current market prices.
- Prices reflect unadjusted historical trading records without explicit dividend or split-adjustment columns.
- Historical performance and volatility describe past behavior and do not guarantee future returns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

raw_df = pd.read_csv('../data/raw/S&P 500 Stock Prices 2014-2017.csv')
print(f'Raw Shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns')
raw_df.head()

## 2. Data Inspection & Quality Audit
We inspect data types, date coverage, duplicate symbol-date pairs, missing OHLC values, zero-volume records, and incomplete stock histories.

In [ ]:
df = raw_df.copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

print('Missing values per column:\n', df.isnull().sum())
print('Duplicate symbol-date pairs:', df.duplicated(subset=['symbol', 'date']).sum())
print('Zero-volume rows:', (df['volume'] == 0).sum())
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
print('Unique symbols:', df['symbol'].nunique(), '| Trading days:', df['date'].nunique())

symbol_counts = df.groupby('symbol')['date'].count()
print('Stocks with full 1,007 days:', (symbol_counts == 1007).sum())
print('Stocks with partial history (< 1,007 days):', (symbol_counts < 1007).sum())

## 3. Data Cleaning & Feature Engineering (Per Symbol)
- Impute the 11 missing `open`/`high`/`low` cells using that trading day's intact `close` price (never `0`), while preserving quality flags (`had_missing_ohl`, `is_zero_volume`).
- Compute `year`, `month`, `quarter`, `daily_price_change`, `daily_return_pct`, `daily_trading_range`, and 20-day / 50-day moving averages strictly grouped by `symbol`.

In [ ]:
df['had_missing_ohl'] = df[['open', 'high', 'low']].isnull().any(axis=1)
df['is_zero_volume'] = df['volume'] == 0
for col in ['open', 'high', 'low']:
    df[col] = df[col].fillna(df['close'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = 'Q' + df['date'].dt.quarter.astype(str)
df['daily_price_change'] = df.groupby('symbol')['close'].diff()
df['daily_return_pct'] = df.groupby('symbol')['close'].pct_change() * 100
df['daily_trading_range'] = df['high'] - df['low']
df['ma_20'] = df.groupby('symbol')['close'].transform(lambda s: s.rolling(20, min_periods=1).mean())
df['ma_50'] = df.groupby('symbol')['close'].transform(lambda s: s.rolling(50, min_periods=1).mean())
df['trading_days_count'] = df.groupby('symbol')['date'].transform('count')
df['is_complete_history'] = df['trading_days_count'] == 1007

summary = pd.read_csv('../data/processed/stock_kpi_summary.csv')
full_summary = summary[summary['is_complete_history']].copy()
full_summary.sort_values('total_return_pct', ascending=False).head(10)

## 4. Comparative Analysis of 6 Focus Stocks & Validation
We compare `NVDA`, `AMZN`, `AAPL`, `BAC`, `JNJ`, and `CHK` across cumulative return, daily volatility, closing price range, and average trading volume, followed by independent calculation verification.

In [ ]:
focus_tickers = ['NVDA', 'AMZN', 'AAPL', 'BAC', 'JNJ', 'CHK']
focus_comp = full_summary[full_summary['symbol'].isin(focus_tickers)].set_index('symbol').loc[focus_tickers]
print(focus_comp[['start_close', 'end_close', 'min_close', 'max_close', 'total_return_pct', 'volatility_pct', 'avg_daily_volume']].to_string())

# Independent Validation Check
aapl = df[df['symbol'] == 'AAPL'].head(3)
manual_ret = ((aapl.iloc[1]['close'] - aapl.iloc[0]['close']) / aapl.iloc[0]['close']) * 100
assert np.isclose(manual_ret, aapl.iloc[1]['daily_return_pct']), 'Return mismatch!'
print(f'\nValidation Passed: AAPL 2014-01-03 Daily Return = {manual_ret:.6f}%')